In [4]:
import torch

n = 2
C = 1
H = 7
W = 7
oH = 2
oW = 2
L = 4
input = torch.rand(n, C, H, W)
boxes = [torch.zeros(L, 4) for _ in range(n)]
for i in range(n):
  boxes[i][:, 0] = torch.rand(L) * (H-oH)       # y
  boxes[i][:, 1] = torch.rand(L) * (W-oW)       # x
  boxes[i][:, 2] = oH + torch.rand(L) * (H-oH)  # w
  boxes[i][:, 3] = oW + torch.rand(L) * (W-oW)  # h

  boxes[i][:,2:] += boxes[i][:,:2]
  boxes[i][:,2] = torch.clamp(boxes[i][:,2], max=H-1)
  boxes[i][:,3] = torch.clamp(boxes[i][:,3], max=W-1)
output_size = (oH, oW)

In [5]:
print("input dimentions: (n, C, H, W) =", input.shape)
print("boxes dimentions: (n, L, 4) =", len(boxes), boxes[0].shape)
print("output_size: (oH, oW) =", output_size)

input dimentions: (n, C, H, W) = torch.Size([2, 1, 7, 7])
boxes dimentions: (n, L, 4) = 2 torch.Size([4, 4])
output_size: (oH, oW) = (2, 2)


In [6]:
print("input:")
print(input)
print("boxes:")
print(boxes)

input:
tensor([[[[5.7514e-01, 7.1932e-01, 1.8840e-01, 4.1528e-01, 1.0918e-01,
           8.0226e-01, 8.7219e-01],
          [4.4276e-01, 6.2362e-01, 8.2646e-01, 6.9508e-01, 4.0983e-01,
           8.7308e-02, 5.2021e-01],
          [3.7526e-01, 9.2274e-02, 3.4476e-01, 9.2405e-01, 6.9576e-01,
           8.6231e-01, 6.1113e-02],
          [6.1781e-01, 9.1124e-01, 1.4141e-02, 6.1978e-01, 2.9038e-01,
           9.9290e-01, 4.2525e-01],
          [4.8051e-01, 5.3036e-01, 4.1635e-01, 8.8289e-01, 8.6339e-03,
           3.0899e-01, 7.2657e-01],
          [3.7331e-01, 9.8198e-01, 1.2152e-01, 7.5717e-01, 6.8176e-01,
           3.4851e-01, 4.4051e-01],
          [4.6605e-01, 6.9886e-04, 7.6966e-01, 2.1726e-01, 4.4059e-01,
           2.2766e-01, 6.8707e-01]]],


        [[[3.9954e-01, 4.2023e-01, 5.8798e-01, 9.3012e-01, 9.7601e-01,
           3.1933e-01, 9.2403e-01],
          [6.6128e-01, 7.0092e-01, 7.0242e-01, 5.7442e-02, 3.7674e-01,
           5.0274e-01, 9.5807e-01],
          [1.5163e-01, 7.1

In [7]:
def roi_pooling(input, boxes, output_size):
    """
    input: (N, C, H, W)
    boxes: list of length N, each element is (L, 4) tensor (y1, x1, y2, x2)
    output_size: (out_h, out_w)
    
    returns:
        output: (N, L, C, out_h, out_w)
    """
    
    
    N, C, H, W = input.shape
    oH, oW = output_size
    L = boxes[0].shape[0]
    
    # 1. Round the coordinates of the boxes
    boxes_tensor = torch.stack(boxes)  # (N, L, 4)
    boxes_tensor = torch.round(boxes_tensor)

    # 2. Extract y1, x1, y2, x2 from the boxes
    y1 = boxes_tensor[:, :, 0:1]
    x1 = boxes_tensor[:, :, 1:2]
    y2 = boxes_tensor[:, :, 2:3]
    x2 = boxes_tensor[:, :, 3:4]

    
    # 3. Create a grid of (i, j) coordinates for the output size
    i = torch.arange(oH, dtype=torch.float32).view(1, 1, oH)
    j = torch.arange(oW, dtype=torch.float32).view(1, 1, oW)

    # 5. Calculate the boundaries for all boxes in the batch in parallel.
    box_H = y2 - y1 + 1
    box_W = x2 - x1 + 1

    y_start = torch.floor(y1 + i * box_H / oH).int()
    y_end   = torch.ceil(y1 + (i + 1) * box_H / oH).int()
    
    x_start = torch.floor(x1 + j * box_W / oW).int()
    x_end   = torch.ceil(x1 + (j + 1) * box_W / oW).int()

    # 6. Max pool over the regions defined by these boundaries for each box and each channel.
    out = torch.zeros((N, L, C, oH, oW), dtype=input.dtype)
    
    # For every image in the batch
    for n in range(N):
        # Select a box in the image
        for l in range(L):
            # Process every output cell (i, j) for this box
            for i_idx in range(oH):
                for j_idx in range(oW):
                    # Get the specific boundaries for this output cell
                    ys, ye = y_start[n, l, i_idx], y_end[n, l, i_idx]
                    xs, xe = x_start[n, l, j_idx], x_end[n, l, j_idx]
                    
                    # Security check: verify that the region is valid (non-empty)
                    if ye > ys and xe > xs:
                        # Extract the patch from the input feature map
                        patch = input[n, :, ys:ye, xs:xe]
                        
                        # Eseguiamo il max pooling per canale.
                        # Appiattiamo le dimensioni spaziali (C, -1) e prendiamo il massimo (dim=1)
                        out[n, l, :, i_idx, j_idx] = patch.reshape(C, -1).max(dim=1)[0]
    return out

In [8]:
out = roi_pooling(input, boxes, output_size)
print("Result of Roi Pooling:")
print(out)

Result of Roi Pooling:
tensor([[[[[0.8623, 0.8623],
           [0.9929, 0.9929]]],


         [[[0.9929, 0.9929],
           [0.6818, 0.7266]]],


         [[[0.9929, 0.9929],
           [0.6818, 0.6871]]],


         [[[0.8265, 0.9240],
           [0.9112, 0.9240]]]],



        [[[[0.9848, 0.8189],
           [0.9848, 0.6826]]],


         [[[0.9592, 0.9592],
           [0.9514, 0.9848]]],


         [[[0.9373, 0.9848],
           [0.9514, 0.7451]]],


         [[[0.9141, 0.9592],
           [0.9514, 0.9848]]]]])
